> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board, uploads the
> package and talks to the TCA9548A mux and the TMP102 on I2C bus 1 (read-only transfers, plus
> selecting and releasing mux channel 2). *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host
>   against `micropython_i2c.sim.bus.FakeBus`, writes the module and its tests, and needs no
>   board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

Notebook-only: import the package before anything else. Its `__init__` imports `node`, and
the header cell below rewrites `node.py`; importing the package first means no later import
reads the half-written file.

In [ ]:
import micropython_i2c  # noqa: F401

# Hardware and configuration

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus`, and a recording stand-in connection | `machine.I2C(1)`, **bus 1**, scl=15 sda=16, 100 kHz (`bus_open("feathers3", 1)`) |
| **Mux** | `FakeMux` at `0x71` | SparkFun Qwiic Mux (TCA9548A) at `0x70`; RESET line not used |
| **Parts used** | `FakeRegisterPart` at `0x35` | TMP102 at `0x48` on mux channel 2: configuration register read, never written |
| **Result** | not run yet | not run yet |

# TODO:
Nothing open.

Decided (2026-11-16): the node tree moved from a hand-written file into this notebook. Its
public API and behaviour are unchanged: every driver depends on them.

# Python file
location: `src/micropython_i2c/node.py`

tests: `tests/test_node.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/node.py
"""The node tree: ``I2CNode`` -> {``I2CDevice``, ``I2CMux``}.

``I2CNode`` holds two contracts:

* **Datasheet facts** as class attributes (``ALLOWED_ADDRS``, ``BAUD_*``), so config
  validation and scan matching can reason about a part before any instance exists.
* **Transport**: four methods (``read_mem``, ``write_mem``, ``read_raw``,
  ``write_raw``). The protocol mixins in ``protocol.py`` call only these, never
  ``self.conn`` or a bus, which is what lets the same mixin serve a sensor behind three
  muxes and a mux on the bare bus.

Both node types reach the hardware through an ``I2CConnection``. A device is handed one
by the manager. A mux builds its own from its *upstream* path (the hops that reach the
mux itself), so it never routes through itself.

Generated by ``dev_notebooks/node_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None. `machine.Pin` is imported lazily inside `I2CMux.__init__`, only when a reset pin is
given, so the module imports on the host.

## Internal
`sleep_us` for the reset pulse, `I2CConnection` for the mux's own connection.

In [ ]:
%%extract ../src/micropython_i2c/node.py -a
from micropython_i2c.compat import sleep_us
from micropython_i2c.connection import I2CConnection

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_node.py
import pytest

from micropython_i2c.compat import NullLock
from micropython_i2c.connection import I2CConnection
from micropython_i2c.node import I2CDevice, I2CMux, I2CNode
from micropython_i2c.sim.bus import FakeBus, FakeMux, FakeRegisterPart

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

from micropython_i2c.compat import NullLock
from micropython_i2c.sim.bus import FakeBus, FakeMux, FakeRegisterPart

# Intent
**Purpose:** the base classes every part derives from. A node knows two things: the
datasheet facts about its part (class attributes, readable with no instance) and how to move
bytes (four transport methods over its `I2CConnection`).

```mermaid
classDiagram
    I2CNode <|-- I2CDevice
    I2CNode <|-- I2CMux
    I2CNode : ALLOWED_ADDRS, BAUD_*, REG_ADDR_BITS
    I2CNode : conn
    I2CNode : read_mem() write_mem() read_raw() write_raw()
    I2CMux : CHANNELS
    I2CMux : channel_select() channel_deselect() control_read() reset()
```

**Why four transport methods.** The protocol mixins (`RegisterI2CBase`, `CommandI2CBase`)
call only these. They never see a bus, an address or a lock, so the same mixin serves a
sensor three muxes deep and a mux on the bare bus.

**Device vs mux.** A device is handed a connection by the manager. A mux builds its own, from
the *upstream* path that reaches the mux itself, so a mux never routes through itself.

**Used by:** every driver and mux in `devices`, the manager, the routing table (class
attributes only).

## What the module defines

| Name | What it is |
|---|---|
| `I2CNode` | datasheet facts as class attributes, plus the four-method transport |
| `I2CDevice` | a routed device, given its connection; the base of ordinary drivers |
| `I2CMux` | a routing element with its own upstream connection, the mux API and `reset` |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/node.py -a
__all__ = ["I2CDevice", "I2CMux", "I2CNode"]

## The class hierarchy
`I2CNode` is the common base. `I2CDevice` adds nothing: it exists so drivers say what they
are, and so the manager can tell devices from muxes. `I2CMux` builds its own connection and
declares the mux API that each mux driver implements.

## `I2CNode`
Class attributes hold the datasheet facts; a driver overrides them. `addr_ok` and `baud_ok`
are class methods so the routing table can check a config entry before anything is built.
Extra keyword arguments from a config entry (labels, calibration hints) are kept on
`self.kwargs`, so the base needs to know nothing of them.

| Attribute | Default | Meaning |
|---|---|---|
| `ALLOWED_ADDRS` | `()` | every address the part can be strapped to |
| `BAUD_NOMINAL` | `100_000` | normal operating speed |
| `BAUD_MAX` | `400_000` | datasheet ceiling; a shared bus runs at the lowest one |
| `BAUD_MIN` | `10_000` | datasheet floor |
| `REG_ADDR_BITS` | `8` | register-number width on the wire (16 for the VL53L4CD) |

In [ ]:
%%extract ../src/micropython_i2c/node.py -a
class I2CNode:
    """Common ancestor for anything addressable on an I2C bus.

    Class attributes (override from the datasheet):
        ALLOWED_ADDRS: Every 7-bit address the part can be strapped to.
        BAUD_NOMINAL: Normal operating speed.
        BAUD_MAX: Datasheet ceiling. A shared bus runs at or below the lowest one.
        BAUD_MIN: Datasheet floor. Some parts time out if clocked too slowly.
        REG_ADDR_BITS: Width of a register number on the wire: 8, or 16 for parts with
            a 16-bit register map (VL53L4CD).

    Attributes:
        conn: The ``I2CConnection`` used for every transfer.
        kwargs: Extra keyword arguments from the config entry (labels, calibration
            hints), kept rather than rejected so the base needs to know nothing of them.
    """

    ALLOWED_ADDRS = ()
    BAUD_NOMINAL = 100_000
    BAUD_MAX = 400_000
    BAUD_MIN = 10_000
    REG_ADDR_BITS = 8

    def __init__(self, conn, **kwargs):
        self.conn = conn
        self.kwargs = kwargs

    def __repr__(self):
        return "<{} {!r}>".format(type(self).__name__, self.conn)

    @classmethod
    def addr_ok(cls, addr):
        """True if ``addr`` is one this part can answer on. No instance needed."""
        return addr in cls.ALLOWED_ADDRS

    @classmethod
    def baud_ok(cls, freq):
        """True if ``freq`` is inside this part's datasheet limits."""
        return cls.BAUD_MIN <= freq <= cls.BAUD_MAX

    @property
    def addr(self):
        """This node's 7-bit address (from the connection)."""
        return self.conn.addr

    # Transport contract. Protocol mixins are written against exactly these four.

    def read_mem(self, reg, nbytes):
        """Read ``nbytes`` from register ``reg``."""
        return self.conn.readfrom_mem(reg, nbytes, self.REG_ADDR_BITS)

    def write_mem(self, reg, data):
        """Write ``data`` to register ``reg``."""
        self.conn.writeto_mem(reg, data, self.REG_ADDR_BITS)

    def read_raw(self, nbytes):
        """Read ``nbytes`` with no register phase."""
        return self.conn.readfrom(nbytes)

    def write_raw(self, data):
        """Write raw bytes with no register phase."""
        self.conn.writeto(data)

### validation/test
A stand-in connection for the cells below: it records every call and returns zeros, so the
tests see exactly what the transport methods pass on, `addrsize` included (`FakeBus` does not
log it).

In [ ]:
%%extract ../tests/test_node.py -a
class RecordingConn:
    """Stands in for an ``I2CConnection``: records each call, reads return zeros."""

    def __init__(self, addr=0x48):
        self.addr = addr
        self.calls = []

    def __repr__(self):
        return "<RecordingConn {:#04x}>".format(self.addr)

    def readfrom_mem(self, reg, nbytes, addrsize=8):
        self.calls.append(("readfrom_mem", reg, nbytes, addrsize))
        return bytes(nbytes)

    def writeto_mem(self, reg, data, addrsize=8):
        self.calls.append(("writeto_mem", reg, bytes(data), addrsize))

    def readfrom(self, nbytes):
        self.calls.append(("readfrom", nbytes))
        return bytes(nbytes)

    def writeto(self, data):
        self.calls.append(("writeto", bytes(data)))
        return len(data)

A demo part with its datasheet facts filled in, and a 16-bit-register variant. They derive
from `I2CNode` directly, since `I2CDevice` is defined below.

In [ ]:
%%extract ../tests/test_node.py -a
class DemoPart(I2CNode):
    """A part strappable to 0x48..0x4B, 1 kHz..400 kHz."""

    ALLOWED_ADDRS = (0x48, 0x49, 0x4A, 0x4B)
    BAUD_MIN = 1_000


class WidePart(DemoPart):
    """The same part with a 16-bit register map."""

    REG_ADDR_BITS = 16

#### validation: nominal usage
Datasheet checks without an instance, then each transport method and what reaches the
connection.

In [ ]:
print("addr_ok(0x49):", DemoPart.addr_ok(0x49), "| baud_ok(100 kHz):", DemoPart.baud_ok(100_000))
conn = RecordingConn(0x49)
dev = DemoPart(conn, label="bench")
print(dev, "| addr:", hex(dev.addr), "| kwargs:", dev.kwargs)
dev.read_mem(0x01, 2)
dev.write_mem(0x01, b"\x60\xa0")
dev.read_raw(3)
dev.write_raw(b"\x2c\x06")
WidePart(conn).read_mem(0x010F, 1)
for call in conn.calls:
    print("   ", call)

#### validation: anti-nominal usage
None: `I2CNode` raises nothing of its own. A wrong address or speed is answered `False`, and
a transfer error comes from the connection unchanged.

In [ ]:
print("addr_ok(0x50):", DemoPart.addr_ok(0x50), "| baud_ok(1 MHz):", DemoPart.baud_ok(1_000_000))
print("base class allows no address:", I2CNode.addr_ok(0x48))

#### unit

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_datasheet_defaults():
    assert I2CNode.ALLOWED_ADDRS == ()
    assert (I2CNode.BAUD_MIN, I2CNode.BAUD_NOMINAL, I2CNode.BAUD_MAX) == (10_000, 100_000, 400_000)
    assert I2CNode.REG_ADDR_BITS == 8
    assert not I2CNode.addr_ok(0x48)

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_addr_ok_and_baud_ok_need_no_instance():
    assert DemoPart.addr_ok(0x4B) and not DemoPart.addr_ok(0x4C)
    assert DemoPart.baud_ok(1_000) and DemoPart.baud_ok(400_000)
    assert not DemoPart.baud_ok(999) and not DemoPart.baud_ok(400_001)

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_addr_repr_and_kwargs():
    dev = DemoPart(RecordingConn(0x4A), label="left", offset=0.5)
    assert dev.addr == 0x4A
    assert repr(dev) == "<DemoPart <RecordingConn 0x4a>>"
    assert dev.kwargs == {"label": "left", "offset": 0.5}

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_transport_methods_pass_reg_addr_bits():
    conn = RecordingConn()
    DemoPart(conn).read_mem(0x01, 2)
    DemoPart(conn).write_mem(0x02, b"\x12")
    WidePart(conn).read_mem(0x010F, 1)
    WidePart(conn).write_mem(0x0110, b"\x34")
    assert conn.calls == [
        ("readfrom_mem", 0x01, 2, 8),
        ("writeto_mem", 0x02, b"\x12", 8),
        ("readfrom_mem", 0x010F, 1, 16),
        ("writeto_mem", 0x0110, b"\x34", 16),
    ]

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_raw_transport_has_no_register_phase():
    conn = RecordingConn()
    dev = DemoPart(conn)
    assert dev.read_raw(3) == b"\x00\x00\x00"
    assert dev.write_raw(b"\x2c\x06") is None  # the ACK count is not passed on
    assert conn.calls == [("readfrom", 3), ("writeto", b"\x2c\x06")]

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_device_reads_through_a_routed_connection():
    bus = FakeBus()
    bus.attach(0x71, FakeMux())
    bus.attach(0x48, FakeRegisterPart({0x01: 0x60, 0x02: 0xA0}), path=[(0x71, 2)])
    dev = DemoPart(I2CConnection(bus, 0x48, [[0x71, 2]], NullLock()))
    assert dev.read_mem(0x01, 2) == b"\x60\xa0"

## `I2CDevice`
A routed device, reached through a connection the manager built for it. It adds nothing to
`I2CNode`; ordinary drivers subclass it alongside a protocol mixin.

In [ ]:
%%extract ../src/micropython_i2c/node.py -a
class I2CDevice(I2CNode):
    """A routed device, reached through a connection the manager built for it.

    The base class for ordinary drivers: subclass it alongside a protocol mixin,
    override the datasheet attributes and add device methods::

        class MySensor(RegisterI2CBase, I2CDevice):
            ALLOWED_ADDRS = (0x48, 0x49)

    Args:
        conn: A fully bound ``I2CConnection`` to this device.
        **kwargs: Kept on ``self.kwargs``.
    """

### validation/test
The behaviour is `I2CNode`'s, validated above. What is left to check: a driver built on
`I2CDevice` is an `I2CNode`, not an `I2CMux`, which is how the manager tells them apart.

#### validation: nominal usage

In [ ]:
class DemoDevice(I2CDevice):
    ALLOWED_ADDRS = (0x48,)


demo = DemoDevice(RecordingConn(), label="x")
print(demo, "| node:", isinstance(demo, I2CNode), "| addr_ok(0x48):", DemoDevice.addr_ok(0x48))

#### validation: anti-nominal usage
None: `I2CDevice` adds no behaviour that could fail.

#### unit

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_device_is_a_plain_node():
    dev = I2CDevice(RecordingConn(0x4A), label="x")
    assert isinstance(dev, I2CNode)
    assert dev.addr == 0x4A and dev.kwargs == {"label": "x"}
    assert repr(dev) == "<I2CDevice <RecordingConn 0x4a>>"
    assert I2CDevice.ALLOWED_ADDRS == () and not hasattr(dev, "channel_select")

## `I2CMux`
A routing element. It builds its own `I2CConnection` from the hops that reach the mux itself
(`path`), with the bus's shared lock: a mux with a lock of its own could re-route the bus in
the middle of another connection's transfer.

The mux API (`channel_select`, `channel_deselect`, `control_read`) raises
`NotImplementedError` here: channel encoding is part-specific, so each mux driver implements
it. `channel_check` bounds-checks against `CHANNELS`.

`reset` pulses the active-low RESET pin, if one was given: the recovery of last resort when a
downstream part holds SDA low. `machine.Pin` is imported only when a pin is given, so the
class builds on the host without one.

In [ ]:
%%extract ../src/micropython_i2c/node.py -a
class I2CMux(I2CNode):
    """A routing element. Talks to the bus at its own address via its upstream path.

    Subclasses implement the mux API the manager relies on (``channel_select``,
    ``channel_deselect``, ``control_read``), since channel encoding is part-specific.

    Args:
        bus: The ``machine.I2C`` object.
        address: The mux's own 7-bit address.
        lock: The per-bus mutex shared with every connection on the bus. Must be the
            shared one: a mux locking independently could re-route the bus mid-transfer.
        path: Hops needed to reach *this* mux, for chained muxes. Empty for a mux wired
            straight to the bus.
        reset_pin: Pin id of the active-low RESET line, if wired.
        **kwargs: Kept on ``self.kwargs``.

    Class attributes:
        CHANNELS: Downstream channel count; the routing table bounds-checks against it.
    """

    CHANNELS = 0

    def __init__(self, bus, address, lock=None, path=None, reset_pin=None, **kwargs):
        super().__init__(I2CConnection(bus, address, path, lock), **kwargs)
        self.reset_pin = None
        if reset_pin is not None:
            # Lazy import keeps this module importable off-target (host tests, lint).
            from machine import Pin

            self.reset_pin = Pin(reset_pin, Pin.OUT, value=1)  # active low: idle high

    def channel_check(self, channel):
        """Raise ``ValueError`` unless ``channel`` is in ``0..CHANNELS-1``."""
        if not 0 <= channel < self.CHANNELS:
            raise ValueError("channel {} out of range 0..{}".format(channel, self.CHANNELS - 1))

    def channel_select(self, channel):
        """Route the mux to ``channel``, deselecting any others."""
        raise NotImplementedError

    def channel_deselect(self):
        """Disconnect every channel, isolating everything downstream."""
        raise NotImplementedError

    def control_read(self):
        """Read back the current channel selection."""
        raise NotImplementedError

    def reset(self):
        """Pulse the RESET pin. Returns False if no pin was wired.

        The recovery of last resort: a downstream part holding SDA low wedges the
        segment and cannot be talked out of it. Resetting the mux drops every channel
        and cuts it off. The 10 us dwells are comfortably over TCA9548A-class minimums.
        """
        if self.reset_pin is None:
            return False
        self.reset_pin.value(0)
        sleep_us(10)
        self.reset_pin.value(1)
        sleep_us(10)
        return True

### validation/test
A demo mux for the cells below: eight channels, one-hot control byte (the TCA9548A's
encoding), written through the node's own transport. And a stand-in pin that records the
levels it is driven to, for `reset`.

In [ ]:
%%extract ../tests/test_node.py -a
class DemoMux(I2CMux):
    """Eight channels, one bit each in a single control byte."""

    ALLOWED_ADDRS = tuple(range(0x70, 0x78))
    CHANNELS = 8

    def channel_select(self, channel):
        self.channel_check(channel)
        self.write_raw(bytes([1 << channel]))

    def channel_deselect(self):
        self.write_raw(b"\x00")

    def control_read(self):
        return self.read_raw(1)[0]


class RecordingPin:
    """Stands in for ``machine.Pin``: ``levels`` lists every value driven."""

    def __init__(self):
        self.levels = []

    def value(self, v):
        self.levels.append(v)

#### validation: nominal usage
A mux at `0x71` on the bare bus, with a part behind channel 2; a second mux chained behind it
builds its connection from the upstream path.

In [ ]:
bus = FakeBus()
bus.attach(0x71, FakeMux())
bus.attach(0x48, FakeRegisterPart(), path=[(0x71, 2)])
shared = NullLock()
mux = DemoMux(bus, 0x71, shared, label="main")
print(mux, "| lock shared:", mux.conn.lock is shared, "| kwargs:", mux.kwargs)
mux.channel_select(2)
print("control:", hex(mux.control_read()), "| scan:", [hex(a) for a in bus.scan()])
mux.channel_deselect()
print("control:", hex(mux.control_read()), "| scan:", [hex(a) for a in bus.scan()])
print("chained:", DemoMux(bus, 0x72, shared, path=[[0x71, 5]]))
print("reset without a pin:", mux.reset())

#### validation: anti-nominal usage
A channel past `CHANNELS` is refused before anything reaches the bus.

In [ ]:
n = len(bus.log)
try:
    mux.channel_select(8)  # channels are 0..7
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("channel 8 should have raised ValueError")
finally:
    print("bus untouched:", len(bus.log) == n)

The base class does not know a mux's encoding.

In [ ]:
try:
    I2CMux(bus, 0x71).channel_select(0)  # the base has no channel encoding
except NotImplementedError:
    print("raised NotImplementedError, as expected:\n    (no message)")
else:
    raise AssertionError("I2CMux.channel_select should have raised NotImplementedError")

#### unit

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_mux_builds_its_own_upstream_connection():
    bus, shared = FakeBus(), NullLock()
    mux = DemoMux(bus, 0x72, shared, path=[[0x71, 5]], label="inner")
    assert isinstance(mux.conn, I2CConnection)
    assert (mux.conn.bus, mux.conn.addr, mux.conn.path) == (bus, 0x72, [[0x71, 5]])
    assert mux.conn.lock is shared
    assert mux.addr == 0x72 and mux.kwargs == {"label": "inner"}
    assert DemoMux(bus, 0x70).conn.path == []

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_mux_selects_through_its_transport():
    bus = FakeBus()
    bus.attach(0x71, FakeMux())
    bus.attach(0x48, FakeRegisterPart(), path=[(0x71, 2)])
    mux = DemoMux(bus, 0x71, NullLock())
    mux.channel_select(2)
    assert mux.control_read() == 0x04 and 0x48 in bus.scan()
    mux.channel_deselect()
    assert mux.control_read() == 0 and bus.scan() == [0x71]

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_channel_check_bounds():
    mux = DemoMux(FakeBus(), 0x70)
    mux.channel_check(0)
    mux.channel_check(7)
    for bad in (-1, 8):
        with pytest.raises(ValueError, match="channel {} out of range 0..7".format(bad)):
            mux.channel_check(bad)
    with pytest.raises(ValueError, match="out of range 0..-1"):
        I2CMux(FakeBus(), 0x70).channel_check(0)  # the base has no channels

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_base_mux_api_is_abstract():
    mux = I2CMux(FakeBus(), 0x70)
    for call in (lambda: mux.channel_select(0), mux.channel_deselect, mux.control_read):
        with pytest.raises(NotImplementedError):
            call()

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_reset_without_pin_returns_false():
    mux = DemoMux(FakeBus(), 0x70)
    assert mux.reset_pin is None
    assert mux.reset() is False

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_reset_pulses_the_pin_low_then_high():
    mux = DemoMux(FakeBus(), 0x70)
    mux.reset_pin = RecordingPin()  # what __init__ builds from reset_pin= on the board
    assert mux.reset() is True
    assert mux.reset_pin.levels == [0, 1]

In [ ]:
%%extract ../tests/test_node.py -a --strip-ipytest
%%ipytest


def test_reset_pin_needs_machine():
    with pytest.raises(ImportError):
        DemoMux(FakeBus(), 0x70, reset_pin=5)  # no machine module on the host

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3, uploads the
> package, and on I2C bus 1 selects and releases channel 2 of the TCA9548A at `0x70` and reads
> the TMP102 at `0x48` (configuration register). Nothing is written to the TMP102, and the
> mux's RESET line is not touched.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

**Hardware and configuration** (same as the table at the top):

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus`, and a recording stand-in connection | `machine.I2C(1)`, **bus 1**, scl=15 sda=16, 100 kHz (`bus_open("feathers3", 1)`) |
| **Mux** | `FakeMux` at `0x71` | SparkFun Qwiic Mux (TCA9548A) at `0x70`; RESET line not used |
| **Parts used** | `FakeRegisterPart` at `0x35` | TMP102 at `0x48` on mux channel 2: configuration register read, never written |
| **Result** | not run yet | not run yet |

Layout: connect and set up, the **matching tests** (one per unit test above that makes sense
on the board, same name), then the tally and disconnect. `test_reset_pulses_the_pin_low_then_high`
has no match: the bench's RESET line is not wired to a known pin.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c.boards import bus_open
from micropython_i2c.compat import default_lock
from micropython_i2c.connection import I2CConnection, path_release
from micropython_i2c.debug import check, check_raises, check_reset, check_summary
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.node import I2CDevice, I2CMux, I2CNode

check_reset()

Notebook-only: the bus, the bench constants and the demo classes: the TMP102 as a bare
`I2CDevice`, and the TCA9548A as a minimal `I2CMux` (one-hot control byte), not the real
drivers, so only this module is under test. The mux starts deselected. If the bench is
rewired, change the constants.

**What to do:** nothing.

In [ ]:
BUS_ID, MUX_ADDR, TMP_ADDR = 1, 0x70, 0x48
TMP_PATH = [[MUX_ADDR, 2]]
TMP_REG_CONFIG = 0x01

bus = bus_open("feathers3", BUS_ID, freq=100_000)
bus_lock = default_lock()
path_release(bus, TMP_PATH)


class BenchTMP(I2CDevice):
    """The TMP102 as a bare device: datasheet addresses only."""

    ALLOWED_ADDRS = (0x48, 0x49, 0x4A, 0x4B)


class BenchMux(I2CMux):
    """The TCA9548A as a minimal mux: eight channels, one-hot control byte."""

    ALLOWED_ADDRS = tuple(range(0x70, 0x78))
    CHANNELS = 8

    def channel_select(self, channel):
        self.channel_check(channel)
        self.write_raw(bytes([1 << channel]))

    def channel_deselect(self):
        self.write_raw(b"\x00")

    def control_read(self):
        return self.read_raw(1)[0]


print("scan, mux released:", [hex(a) for a in bus.scan()])

## Matching tests

**Matches** `test_datasheet_defaults` and `test_addr_ok_and_baud_ok_need_no_instance`.

**What to do:** nothing.

In [ ]:
check("test_datasheet_defaults", I2CNode.ALLOWED_ADDRS == () and I2CNode.REG_ADDR_BITS == 8
      and (I2CNode.BAUD_MIN, I2CNode.BAUD_NOMINAL, I2CNode.BAUD_MAX) == (10_000, 100_000, 400_000))  # fmt: skip
check("test_addr_ok_and_baud_ok_need_no_instance",
      BenchTMP.addr_ok(0x48) and not BenchTMP.addr_ok(0x4C)
      and BenchTMP.baud_ok(100_000) and not BenchTMP.baud_ok(1_000_000))  # fmt: skip

**Matches** `test_addr_repr_and_kwargs` and `test_device_reads_through_a_routed_connection`:
the TMP102 through channel 2. Its configuration register's resolution bits (`R1 R0`, bits
14..13) always read `11`.

**What to do:** nothing.

In [ ]:
tmp = BenchTMP(I2CConnection(bus, TMP_ADDR, TMP_PATH, bus_lock), label="bench")
print(tmp)
check("test_addr_repr_and_kwargs", tmp.addr == TMP_ADDR and tmp.kwargs == {"label": "bench"}
      and repr(tmp) == "<BenchTMP <I2CConnection 0x48 via 0x70:2>>")  # fmt: skip
cfg = tmp.read_mem(TMP_REG_CONFIG, 2)
check("test_device_reads_through_a_routed_connection", cfg[0] & 0x60 == 0x60,
      "config {:#06x}".format(int.from_bytes(cfg, "big")))  # fmt: skip

**Matches** `test_mux_builds_its_own_upstream_connection`.

**What to do:** nothing.

In [ ]:
mux = BenchMux(bus, MUX_ADDR, bus_lock, label="mux")
check("test_mux_builds_its_own_upstream_connection",
      mux.conn.bus is bus and mux.conn.addr == MUX_ADDR and mux.conn.path == []
      and mux.conn.lock is bus_lock and mux.kwargs == {"label": "mux"})  # fmt: skip

**Matches** `test_mux_selects_through_its_transport` and
`test_raw_transport_has_no_register_phase`: `write_raw` and `read_raw` on the real mux.

**What to do:** nothing.

In [ ]:
mux.channel_select(2)
check("select: control reads 0x04", mux.control_read() == 0x04)
check("select: 0x48 answers a scan", TMP_ADDR in bus.scan())
mux.channel_deselect()
check("deselect: control reads 0x00", mux.control_read() == 0x00)
check("deselect: 0x48 gone from the scan", TMP_ADDR not in bus.scan())

**Matches** `test_channel_check_bounds`: refused before anything reaches the bus, so the mux
stays deselected.

**What to do:** nothing.

In [ ]:
check_raises("channel 8 refused", ValueError, mux.channel_select, 8)
check_raises("channel -1 refused", ValueError, mux.channel_select, -1)
check("mux untouched by the refusals", mux.control_read() == 0x00)

**Matches** `test_base_mux_api_is_abstract` and `test_reset_without_pin_returns_false`.

**What to do:** nothing.

In [ ]:
base = I2CMux(bus, MUX_ADDR, bus_lock)
check_raises("base channel_select", NotImplementedError, base.channel_select, 0)
check_raises("base channel_deselect", NotImplementedError, base.channel_deselect)
check_raises("base control_read", NotImplementedError, base.control_read)
check("test_reset_without_pin_returns_false", base.reset() is False)

## Tally and disconnect

Notebook-only: leave the mux deselected, then the PASS/FAIL tally of every check in this
section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
path_release(bus, TMP_PATH)
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))